In [1]:
# Load secrets from a .env file (kept out of git) so no API key lives in this notebook.
import os, pathlib
for _d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path.home() / "Documents/GitHub/biot6900"]:
    _f = _d / ".env"
    if _f.is_file():
        for _line in _f.read_text().splitlines():
            if "=" in _line and not _line.lstrip().startswith("#"):
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip("\"'"))
        break

In [2]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [3]:
MY_GENE = "CYP2C19"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C19


In [4]:
# Q1
#   (a) if MY_FILE exists, read it; otherwise fetch_gene(MY_GENE) and save it
#   (b) pass it through numeric() so scores are numbers, not strings
#   (c) add a `label` column using clinvar_label()
#   (d) run and print the three quality checks described above
#
# Everything you need is already defined. This is about eight lines.

# Q1 (a): load from cache if present, otherwise fetch once and save
if MY_FILE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(MY_FILE, sep="\t")
    print(f"loaded {len(G)} variants from cache: {MY_FILE}")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)
    print(f"saved to {MY_FILE}")

# (b) scores as numbers, (c) ClinVar label
G = numeric(G)
G["label"] = G.clinvar_sig.map(clinvar_label)

# (d) the three quality checks
print("\n--- CHECK 1: count ---")
print(f"{len(G)} variants for {MY_GENE}")
print(G.label.value_counts().to_string())

print("\n--- CHECK 2: anchors ---")
for rs, name in ANCHORS.items():
    if name.startswith(MY_GENE):
        hit = G[G.rsid.astype(str).str.contains(rs, na=False)]
        print(f"{name} ({rs}): {'FOUND' if len(hit) else 'MISSING'}")
        if len(hit):
            print(hit[["rsid", "consequence", "clinvar_sig", "gnomad_af"]].to_string(index=False))

print("\n--- CHECK 3: score columns populated ---")
cov = G[["cadd_phred", "revel", "alphamissense", "gnomad_af"]].notna().mean().mul(100).round(1)
print(cov.to_string())

os.makedirs("images", exist_ok=True)
G.label.value_counts().rename_axis("label").reset_index(name="n").to_csv("images/module5_part3_clinvar_label_counts.csv", index=False)
cov.rename("pct_populated").rename_axis("score_column").reset_index().to_csv("images/module5_part3_score_column_coverage.csv", index=False)

loaded 80 variants from cache: data/CYP2C19.tsv

--- CHECK 1: count ---
80 variants for CYP2C19
label
other    38
VUS      33
B/LB      9

--- CHECK 2: anchors ---
CYP2C19*2 (rs4244285): FOUND
     rsid        consequence   clinvar_sig  gnomad_af
rs4244285 synonymous_variant drug response   0.167953
rs4244285 synonymous_variant drug response        NaN
CYP2C19*17 (rs12248560): FOUND
      rsid           consequence   clinvar_sig  gnomad_af
rs12248560 upstream_gene_variant Benign; other    0.20012

--- CHECK 3: score columns populated ---
cadd_phred       85.0
revel            67.5
alphamissense    67.5
gnomad_af        68.8


In [5]:
# Q2
#   (a) group G by consequence and count non-null scores per tool
#   (b) print the table, plus the overall percentage scored by each tool
#   (c) print which consequence classes have ZERO REVEL scores
#
# Hint: .groupby("consequence") then .apply(lambda s: s.notna().sum())

TOOLS = ["cadd_phred", "revel", "alphamissense"]

# (a) per consequence class: total variants and non-null scores per tool
cov = G.groupby("consequence").apply(
    lambda d: pd.Series({"n": len(d), **{t: d[t].notna().sum() for t in TOOLS}}),
    include_groups=False).sort_values("n", ascending=False).astype(int)

# (b) the table, plus overall percentage scored by each tool
print("Variants scored per tool, by consequence class:\n")
print(cov.to_string())
print("\nOverall % of variants scored:")
print((G[TOOLS].notna().mean() * 100).round(1).to_string())

# (c) consequence classes where REVEL scored nothing
no_revel = cov.index[cov.revel == 0].tolist()
print("\nClasses with ZERO REVEL scores:", no_revel)

os.makedirs("images", exist_ok=True)
cov.rename_axis("consequence").reset_index().to_csv("images/module5_part3_scored_by_consequence.csv", index=False)
(G[TOOLS].notna().mean() * 100).round(1).rename_axis("tool").reset_index(name="pct_scored").to_csv("images/module5_part3_overall_pct_scored.csv", index=False)

Variants scored per tool, by consequence class:

                        n  cadd_phred  revel  alphamissense
consequence                                                
missense_variant       54          53     53             54
synonymous_variant      7           6      0              0
upstream_gene_variant   5           4      0              0
intergenic_region       1           1      0              0
intron_variant          1           1      0              0
splice_donor_variant    1           0      0              0
start_lost              1           1      1              0
stop_gained             1           1      0              0
stop_lost               1           1      0              0

Overall % of variants scored:
cadd_phred       85.0
revel            67.5
alphamissense    67.5

Classes with ZERO REVEL scores: ['synonymous_variant', 'upstream_gene_variant', 'intergenic_region', 'intron_variant', 'splice_donor_variant', 'stop_gained', 'stop_lost']


In [ ]:
# Q3
#   (a) set MAX_AF to a threshold you can defend for YOUR gene
#   (b) report how many variants it removes
#   (c) CRITICAL: list the P/LP or "drug response" variants your filter removes.
#       If it removes a known functional allele, either change the threshold or
#       justify keeping it in your report.
#   (d) for contrast, report what a 0.001 rare-disease filter would have removed
#   (e) set G to the filtered set

# (a) Threshold
MAX_AF = 0.5

# Variants with no gnomAD frequency are KEPT: missing is not the same as common.
def too_common(d, cutoff):
    return d[d.gnomad_af > cutoff]

# (b) what the chosen threshold removes
removed = too_common(G, MAX_AF)
print(f"MAX_AF = {MAX_AF}: removes {len(removed)} of {len(G)} variants "
      f"({G.gnomad_af.isna().sum()} have no AF and are kept)")
cols = ["rsid", "consequence", "clinvar_sig", "label", "gnomad_af"]
print(removed[cols].to_string(index=False))

# (c) CRITICAL: any P/LP or drug-response variants lost?
def functional(d):
    return d[(d.label == "P/LP") | d.clinvar_sig.astype(str).str.contains("drug response", case=False)]

lost = functional(removed)
print(f"\nP/LP or drug-response variants removed by MAX_AF={MAX_AF}: {len(lost)}")
if len(lost): print(lost[cols].to_string(index=False))

# (d) contrast with a rare-disease filter
rare = too_common(G, 0.001)
lost_rare = functional(rare)
print(f"\nRare-disease filter (0.001) would remove {len(rare)} variants, "
      f"including {len(lost_rare)} P/LP or drug-response:")
print(lost_rare[cols].sort_values("gnomad_af").to_string(index=False))
print("  anchors lost:", [a for a in ANCHORS if a in set(lost_rare.rsid.astype(str))])

# (e) apply
G = G[~(G.gnomad_af > MAX_AF)].copy()
print(f"\nG after filtering: {len(G)} variants")

os.makedirs("images", exist_ok=True)
removed[cols].to_csv("images/module5_part3_af_filter_removed_variants.csv", index=False)
lost[cols].to_csv("images/module5_part3_af_filter_lost_functional_variants.csv", index=False)
lost_rare[cols].to_csv("images/module5_part3_rare_disease_filter_lost_functional_variants.csv", index=False)

MAX_AF = 0.5: removes 2 of 80 variants (25 have no AF and are kept)
      rsid        consequence clinvar_sig label  gnomad_af
rs17885098 synonymous_variant      Benign  B/LB   0.919465
 rs3758581   missense_variant      Benign  B/LB   0.952250

P/LP or drug-response variants removed by MAX_AF=0.5: 0

Rare-disease filter (0.001) would remove 19 variants, including 13 P/LP or drug-response:
       rsid           consequence   clinvar_sig label  gnomad_af
 rs28399504            start_lost drug response other   0.002379
  rs4986893           stop_gained drug response other   0.002506
 rs58973490      missense_variant drug response other   0.002886
 rs17884712      missense_variant drug response other   0.003570
 rs17879685      missense_variant drug response other   0.004596
rs113164681 upstream_gene_variant drug response other   0.005526
 rs17878739 upstream_gene_variant drug response other   0.005535
rs111490789 upstream_gene_variant drug response other   0.005551
 rs17882687      misse

In [7]:
# Q4
#   (a) choose SCORE_COL and BANDS and run calibrate() on G
#   (b) print how many labelled variants the calibration rests on
#   (c) print how many bands rest on fewer than 10 variants
#   (d) if your gene is too small to calibrate, say so in the output and take
#       one of the two documented routes above
#
# Suggested bands: CADD [0,10,20,25,30,50]   REVEL [0,.2,.4,.6,.8,1.0]

# (a) CYP2C19 alone has 0 P/LP variants, so it cannot be calibrated on its own.
# Documented route 1: fetch a second gene and pool. DPYD has ~100 P/LP and ~100 B/LB.
POOL_GENE = "DPYD"
POOL_FILE = DATA / f"{POOL_GENE}.tsv"
if POOL_FILE.exists() and not FORCE_REFETCH:
    D = pd.read_csv(POOL_FILE, sep="\t")
else:
    D = fetch_gene(POOL_GENE)
    D.to_csv(POOL_FILE, sep="\t", index=False)
D = numeric(D)
D["label"] = D.clinvar_sig.map(clinvar_label)
D = D[~(D.gnomad_af > MAX_AF)]            # same frequency filter as Q3

P = pd.concat([G, D], ignore_index=True)  # pooled table used ONLY to fit the calibration

# CADD has the best coverage in CYP2C19 (Q2), and scores non-missense classes too
SCORE_COL = "cadd_phred"
BANDS     = [0, 10, 20, 25, 30, 50]

cal = calibrate(P, SCORE_COL, BANDS)
print(cal.to_string())

# (b) how many labelled variants does this rest on?
print(f"\nCalibration rests on {cal.attrs['n_p']} P/LP and {cal.attrs['n_b']} B/LB variants "
      f"({cal.attrs['n_p'] + cal.attrs['n_b']} total), pooled from {MY_GENE} + {POOL_GENE}.")
print(f"  of which from {MY_GENE}: {(G.label == 'P/LP').sum()} P/LP, {(G.label == 'B/LB').sum()} B/LB")

# (c) thin bands
thin = cal[cal.n < 10]
print(f"\nBands resting on fewer than 10 variants: {len(thin)} of {len(cal)}")
if len(thin): print(thin[["n", "n_PLP", "n_BLB", "LR+"]].to_string())

os.makedirs("images", exist_ok=True)
_cal = cal.copy(); _cal.index = _cal.index.astype(str)
_cal.rename_axis("band").to_csv("images/module5_part3_cadd_calibration_bands.csv")

           n  n_PLP  n_BLB   LR+                evidence
band                                                    
(0, 10]   65      0     65  0.00          strong, benign
(10, 20]  24      0     24  0.00          strong, benign
(20, 25]  11      5      6  2.05  no evidence either way
(25, 30]   4      3      1  7.38    moderate, pathogenic
(30, 50]  31     31      0   inf      strong, pathogenic

Calibration rests on 39 P/LP and 96 B/LB variants (135 total), pooled from CYP2C19 + DPYD.
  of which from CYP2C19: 0 P/LP, 7 B/LB

Bands resting on fewer than 10 variants: 1 of 5
          n  n_PLP  n_BLB   LR+
band                           
(25, 30]  4      3      1  7.38


In [8]:
# Q5
#   (a) take the VUS rows from G
#   (b) map each one to the evidence strength of its calibrated band
#   (c) count how many move towards pathogenic, towards benign, and stay put
#   (d) export to pgx_classified_<MY_GENE>.csv with these columns:
#       rsid, gene, consequence, aa_change, your score column, gnomad_af,
#       evidence, provisional_call

# (a) the VUS rows of CYP2C19
vus = G[G.label == "VUS"].copy()

# (b) map each score to its band, then to that band's calibrated evidence strength.
# Bands come from the pooled calibration `cal` in Q4. A VUS with no CADD score, or one
# outside the bands, gets no evidence.
vus["band"] = pd.cut(vus[SCORE_COL], BANDS)
vus["evidence"] = vus.band.map(cal.evidence.to_dict()).astype(object)
vus["evidence"] = vus.evidence.where(vus.evidence.notna(), "not scored")

def provisional(ev):
    if "pathogenic" in ev: return "VUS, leaning pathogenic"
    if "benign" in ev:     return "VUS, leaning benign"
    return "VUS"          # includes "no evidence either way" and "not scored"

vus["provisional_call"] = vus.evidence.map(provisional)

# (c) how many move?
print(f"{len(vus)} VUS in {MY_GENE}\n")
print(vus.evidence.value_counts().to_string())
print()
print(vus.provisional_call.value_counts().to_string())

# (d) export
OUT = f"pgx_classified_{MY_GENE}.csv"
out_cols = ["rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af",
            "evidence", "provisional_call"]
vus[out_cols].to_csv(OUT, index=False)
print(f"\nwrote {OUT} ({len(vus)} rows)")

os.makedirs("images", exist_ok=True)
vus.evidence.value_counts().rename_axis("evidence").reset_index(name="n").to_csv("images/module5_part3_vus_evidence_counts.csv", index=False)
vus.provisional_call.value_counts().rename_axis("provisional_call").reset_index(name="n").to_csv("images/module5_part3_vus_provisional_calls.csv", index=False)
vus[out_cols].to_csv("images/module5_part3_vus_classified.csv", index=False)

33 VUS in CYP2C19

evidence
strong, benign            16
no evidence either way     8
not scored                 6
moderate, pathogenic       3

provisional_call
VUS, leaning benign        16
VUS                        14
VUS, leaning pathogenic     3

wrote pgx_classified_CYP2C19.csv (33 rows)


In [9]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="rsid", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

os.makedirs("images", exist_ok=True)
if G.avi.notna().sum() > 0:
    a.sort_values("avi", ascending=False)[cols].head(10).to_csv("images/module5_part3_atlas_top_variants.csv", index=False)
    a.top_feature.value_counts().rename_axis("top_feature").reset_index(name="n").to_csv("images/module5_part3_atlas_top_features.csv", index=False)
else:
    nc.consequence.value_counts().rename_axis("consequence").reset_index(name="n").to_csv("images/module5_part3_non_missense_consequences.csv", index=False)

  50/78 queried

scored 70 variants (0 failures)

highest-impact CYP2C19 variants by AVI

       rsid          consequence  avi_phred         top_feature  revel  cadd_phred label
  rs4986893          stop_gained      32.43 PROTEIN_TERMINATION    NaN      35.000 other
rs371241592     missense_variant      25.99       ALPHAMISSENSE  0.518      23.600   VUS
 rs72558186 splice_donor_variant      25.51     MERGED_SPLICING    NaN         NaN other
 rs55640102            stop_lost      25.48           STOP_LOST    NaN      13.030 other
 rs41291556     missense_variant      25.07       ALPHAMISSENSE  0.440      25.700 other
rs760289518     missense_variant      24.22       ALPHAMISSENSE  0.506      25.900   VUS
 rs28399504           start_lost      23.84          START_LOST  0.314      15.940 other
rs371309036     missense_variant      23.75       ALPHAMISSENSE  0.407      24.000   VUS
rs372875966     missense_variant      22.93       ALPHAMISSENSE  0.528      23.000   VUS
  rs4244285   synony

In [10]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

os.makedirs("images", exist_ok=True)
d[flags].sum().rename_axis("flag").reset_index(name="n_variants").to_csv("images/module5_part3_discordance_flag_counts.csv", index=False)
hits[cols].to_csv("images/module5_part3_discordant_variants.csv", index=False)

discordant variants in CYP2C19

flag_cadd_vs_revel      3
flag_revel_vs_am        6
flag_scored_but_vus     0
flag_common_but_plp     5
flag_no_tool_at_all    58
flag_atlas_only        21

73 variants trip at least one rule

     rsid           consequence                         consequence_all aa_change  cadd_phred  revel  alphamissense   avi  gnomad_af label
      NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  0.02        NaN   VUS
      NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  6.29        NaN   VUS
      NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN 11.17        NaN   VUS
      NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  9.70        NaN   VUS
      NaN                   NaN                                     NaN       Na